In [1]:
import pandas as pd

In [8]:
raw_data = pd.read_excel('data/Dataset for Data Analytics.xlsx')

In [9]:
raw_data.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [11]:
df = raw_data.copy()

In [17]:
df.isnull().sum()


OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

In [16]:
df['OrderID'].duplicated().sum()

np.int64(0)

In [12]:
df.info()
df.isnull().sum()
df['OrderID'].duplicated().sum()
len(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

1200

In [13]:
print("Rows:", len(df))
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate OrderIDs:", df['OrderID'].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

Rows: 1200

Missing values per column:
OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

Duplicate OrderIDs: 0
Fully duplicated rows: 0


In [18]:
# CR001: blank coupons mean "no coupon used"
df['CouponCode'] = df['CouponCode'].fillna('No Coupon')


# CR003: enforce 2-decimal precision on money columns
df[['UnitPrice', 'TotalPrice']] = df[['UnitPrice', 'TotalPrice']].round(2)

# CR004: trim whitespace on text columns
for c in df.select_dtypes(include=['object', 'string']).columns:
    df[c] = df[c].str.strip()

# Verification gate
assert df['OrderID'].is_unique, "Duplicate OrderIDs found"
assert df['Date'].str.match(r'^\d{4}-\d{2}-\d{2}$').all(), "Bad date format"
assert df.isnull().sum().sum() == 0, "Missing values remain"
assert ((df['Quantity'] * df['UnitPrice']).round(2) == df['TotalPrice']).all()
print("All checks passed. Rows:", len(df))

df.to_csv('data/cleaned_data.csv', index=False)

All checks passed. Rows: 1200


In [19]:
print(df.isnull().sum())
print(df['CouponCode'].value_counts())

OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64
CouponCode
FREESHIP     313
No Coupon    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64


In [20]:
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [21]:
# CR002: confirm dates are valid datetimes
df['Date'] = pd.to_datetime(df['Date'])
assert df['Date'].notna().all()

# ...rest of your cleaning...

# Export with ISO 8601 formatting
df.to_csv('data/cleaned_data.csv', index=False, date_format='%Y-%m-%d')

In [22]:
check = pd.read_csv('data/cleaned_data.csv', dtype=str)
assert check['Date'].str.match(r'^\d{4}-\d{2}-\d{2}$').all()
assert check['OrderID'].is_unique
print("Verified on the exported file")

Verified on the exported file


In [24]:
print("Duplicate OrderIDs:", check['OrderID'].duplicated().sum())
print("Invalid date formats:", (~check['Date'].str.match(r'^\d{4}-\d{2}-\d{2}$')).sum())

Duplicate OrderIDs: 0
Invalid date formats: 0


In [26]:
counts = df['CustomerID'].value_counts()
repeat_ids = counts[counts > 1].index
repeat_rows = df[df['CustomerID'].isin(repeat_ids)]

print("CustomerIDs appearing more than once:", len(repeat_ids))
print("Hidden duplicate rows (ignoring OrderID):",
      repeat_rows.drop(columns='OrderID').duplicated().sum())
print("Repeated IDs with identical date:",
      (repeat_rows.groupby('CustomerID')['Date'].nunique() == 1).sum())
print("Repeated IDs with identical tracking number:",
      (repeat_rows.groupby('CustomerID')['TrackingNumber'].nunique() == 1).sum())
print("Repeated IDs with identical shipping address:",
      (repeat_rows.groupby('CustomerID')['ShippingAddress'].nunique() == 1).sum())

CustomerIDs appearing more than once: 11
Hidden duplicate rows (ignoring OrderID): 0
Repeated IDs with identical date: 0
Repeated IDs with identical tracking number: 0
Repeated IDs with identical shipping address: 0
